# EDA on NHANES 2021-2023 datasets

Here, we will load the NHANES 2021-2023 dataset, then perform some data preprocessing/cleaning and EDA.

## Loading data

Functions to extract data from data directory in current wd, then read as pandas dataframe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import pathlib
from pathlib import Path
from functools import reduce

# Set data directory and base directory
base_dir = Path.cwd().parent
data_dir = base_dir / "data" / "raw"

def load_xpts_as_dfs(data_dir):
    """
    Function to load XPT files in specified dir as a list of pandas dataframes

    Args:
        data_dir : directory containing .XPT files

    Returns:
        df_list : list of pandas dataframes
    """
    # Get filenames in the data directory
    file_names = os.listdir(data_dir)
    file_names = [data_dir / file for file in file_names]

    # Store as dfs 
    df_list = [pd.read_sas(file) for file in file_names]

    return df_list

def data_join(df_list):
    """
    Function to outer-join a list of dataframes using patient ID as join key

    Args:
        df_list : list of pandas dataframes
    
    Returns :
        joined_df : dataframe outer-joined by "SEQN" patient ID column
    """

    # Check that the "SEQN" patient ID column exists
    for i in range(len(df_list)):
        columns = df_list[i].columns

        if "SEQN" not in columns:
            raise KeyError(f"Patient ID column SEQN not present in dataframe {i}")
        
        else:
            # Convert patient ID to int and set as index
            df_list[i]["SEQN"] = df_list[i]["SEQN"].transform(lambda x: int(x))
            df_list[i] = df_list[i].set_index("SEQN")
    
    # Iterated outer_join, using patient ID on index as join key
    joined_df = reduce(lambda df1, df2 :
                       pd.concat([df1, df2], 
                                axis = 1,
                                join = "outer"),
                        df_list)
    
    return joined_df

Run functions on existing .XPT files.

In [111]:
# Load xpt data and check dimensions
df_list = load_xpts_as_dfs(data_dir)
data_df = data_join(df_list)

# Dimensions
print(f"# of individuals : {data_df.shape[0]}")
print(f"# of features    : {data_df.shape[1]}")

# of individuals : 11744
# of features    : 383


## Data cleaning

Our prediction target is asthma, so let's find the columns associated with the medical condition questionnaire ("MCQ_L.xpt"), then remove all but one column ("MCQ010").

In [112]:
# Choose medical condition file
label_file = data_dir / "MCQ_L.xpt"
label_df   = pd.read_sas(label_file)
label_df.set_index("SEQN", inplace = True)

# Select column names other than asthma column
rm_columns = [col for col in list(label_df.columns) if col != "MCQ010"]

# Remove those columns from the data
clean_data = data_df.drop(columns = rm_columns)

# Change label column to asthma
clean_data.rename(columns = {"MCQ010": "Asthma"}, inplace = True)

# Move labels to the last column
clean_data["Asthma"] = clean_data.pop("Asthma")

# Number of features after removing all medical condition labels
print(f"# features after removing labels : {clean_data.shape[1] - 1}")

# features after removing labels : 349


Let's remove all the duplicate columns.

In [122]:
# Select duplicated columns
duplicated = clean_data.columns.duplicated(keep = False)
duplicated_cols = clean_data.columns[duplicated]
duplicated_cols

Index(['WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR',
       'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTSAF2YR', 'WTPH2YR', 'WTPH2YR',
       'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR',
       'WTSAF2YR', 'WTPH2YR', 'WTPH2YR', 'WTPH2YR', 'WTSAF2YR', 'WTPH2YR',
       'WTPH2YR'],
      dtype='str')

The duplicated columns all correspond to phlebotomy weights, which are not needed for this project. So, we can exclude all duplicate columns.

In [124]:
# Remove all duplicated columns
clean_data = clean_data.loc[:, ~duplicated]

# Number of features after removing all phlebotomy weights
print(f"# features after removing phlebotomy weights : {clean_data.shape[1] - 1}")

# features after removing phlebotomy weights : 324


We can convert the Asthma column to categorical labels and remove samples that have no label. According to the NHANES sample data, 1 means yes, 2 means 7, and 9 means, don't know. These codes might show up in other features, so let's define a map to convert these values.

In [ ]:
# 1 is yes, 2 is no, 7 is declined response, 9 is no response
code_to_label = {1: 1, 2: 0, 7: np.nan, 9: np.nan}
clean_data["Asthma"] = clean_data["Asthma"].map(code_to_label)

Now remove all samples without an Asthma label.

In [162]:
# Grab all samples with no label
no_labels = clean_data["Asthma"].isna().to_numpy()

# Subsample
clean_data = clean_data.iloc[~no_labels]

# Number of samples after removing the unlabeled
print(f"# samples after removing unlabeled individuals : {clean_data.shape[0]}")

# samples after removing unlabeled individuals : 11728


Next, figure out how to map the abbreviated ID names to the the actual questions.

Some things to do:
- Check for duplicated columns (some of the labs are the same)
- Check for outlier values
- Deal with NaNs
- Filter out features that have more than 50% NaNs